Synthetic Data generator for generating Siemens Teamcenter PLM object types such as Item, Item Revision, Dataset, Folder and Form 

In [ ]:
import json
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient


In [ ]:
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN:
    print("Successfully loaded HF_TOKEN")
else:
    print("Token not found. Check if your .env file path is correct!")

In [ ]:
client = InferenceClient(
    model="Qwen/Qwen2.5-Coder-32B-Instruct",
    token=HF_TOKEN
)

In [ ]:
DEFINITIONS = {
    "Item": "The Item holds the identity of the object that stays constant throughout its life.",
    "Item Revision": "Tracks a specific snapshot or version of the Item.",
    "Dataset": "describe the file container and its relationship to external software",
    "Folder": "Folders are simpler objects designed primarily for navigation and permissions.",
    "Form" : "Forms are highly variable, but every form contains these structural fields."
}

In [ ]:
EXAMPLES = {
    "Item": {
        "item_id": "A0009899",
        "item_name": "Engine Block",
        "object_type": "Item",
        "owning_user": "Admin",
        "metadata": {
            "material": "Aluminum 6061",
            "weight": "0.45kg"
        }
    },
    "Item Revision":{
        "item_revision_id": "A",
        "release_status_list": ["Released"],
        "date_released": "2026-04-11",
        "last_mod_date": "2026-03-11",
    },

    "Dataset": {
        "dataset_type": "NX Dataset",
        "ref_list": "Part in Teamcenter",
        "tool_useed": "Siemens NX",
        "object_desc": "Contains Design",
    },
   "Folder": {
        "contents": [
            "PROP-100", 
            "PROP-100-REV_A", 
            "SPEC-DOC-01", 
            "AERO_SCHEMA"
        ],
        "object_name": "Propulsion_System_Design",
        "creation_date": "2026-03-01",
        "parent_name": "Main",
    },

    "Form": {
        "form_attribute": "Material_Density_Specification",
        "is_modifiable": "True",
        "validation_rule": "VALUE > 0 AND VALUE < 5000",
        "origin_id": "FRM_99284_B"
    }
}

In [ ]:
def build_prompt(object_type, n_objects):
    definition = DEFINITIONS[object_type]
    example = json.dumps(EXAMPLES[object_type], indent=2)

    system_content = (
        "You are a synthetic data generator. "
        "You output only valid JSON arrays, with no explanation or markdown. "
    )

    user_content = (
        f"Generate {n_objects} samples of type '{object_type}'. "
        f"Definition: {definition}. "
        f"Here is one example sample (Do not copy it, treat it only as an example):\n{example}\n"
        "Return only a JSON array of samples. No explanation, no code blocks, just raw JSON."
    )

    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
    ]

In [ ]:
print(build_prompt(object_type="Item", n_objects=5))

In [ ]:
def generate_data(object_type, n_objects):
    messages = build_prompt(object_type, n_objects)

    response = client.chat_completion(
        messages=messages,
        max_tokens=1024
    )
    response_text = response.choices[0].message.content.strip()
    response_text = response.choices[0].message.content.strip()
    if response_text.startswith("```"):
        response_text = response_text.split("```")[1]
        if response_text.startswith("json"):
            response_text = response_text[4:]
    
    return json.loads(response_text)



In [ ]:
object_type = "Item"
n_objects = 5;
data = generate_data(object_type, n_objects)
print(json.dumps(data, indent=2))